# **Milestone 1**

## **Introduction**

This milestone has been created to familiarize you with the process of Exploratory Data Analysis (EDA), text processing, and establishing baseline similarity metrics required specifically for Natural Language Processing (NLP) pipelines.  

Suggested Readings:
* Tokenization & Text Normalization  

* Stop Words & Vocabulary Constraints  

* TF-IDF (Term Frequency-Inverse Document Frequency)  

* Cosine Similarity in NLP

* Understanding Mean Average Precision (MAP@3)  

Perform the following tasks on the dataset provided as a part of the Kaggle competition.
Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

## **Initialization**

In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")
sample_df = pd.read_csv("/content/sample_submission.csv")

In [3]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## **Question 01**

Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [4]:
freqs = train_df["answer"].value_counts()
print("Frequency distribution:\n", freqs)

most_freq = freqs.max()
least_freq = freqs.min()
print("\nSum of occurrences of most and least frequent option:", most_freq + least_freq)

Frequency distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Sum of occurrences of most and least frequent option: 814


## **Question 02**

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?

In [5]:
cleaned_prompts = train_df["prompt"].astype(str).str.lower()
for p in string.punctuation:
    cleaned_prompts = cleaned_prompts.str.replace(p, "", regex=False)

unique_words = set()
for prompt in cleaned_prompts:
    unique_words.update(prompt.split())

print("Total number of unique words (vocabulary size):", len(unique_words))

Total number of unique words (vocabulary size): 859


## **Question 03**

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [6]:
prompt_row_1 = cleaned_prompts.iloc[0]
words_row_1 = prompt_row_1.split()

filtered_words = [word for word in words_row_1 if word not in ENGLISH_STOP_WORDS]

print("Words left in the prompt for Row ID 1 after filtering:", len(filtered_words))

Words left in the prompt for Row ID 1 after filtering: 13


## **Question 04**

Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [7]:
all_text = []
for col in ["prompt", "A", "B", "C", "D", "E"]:
    all_text.extend(train_df[col].dropna().astype(str).tolist())

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(all_text)

print("Exact total number of feature columns (vocabulary size):", len(tfidf.get_feature_names_out()))

Exact total number of feature columns (vocabulary size): 2762


## **Question 05**

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [8]:
prompt_text = str(train_df["prompt"].iloc[0])
option_a_text = str(train_df["A"].iloc[0])

tfidf_matrix = tfidf.transform([prompt_text, option_a_text])

sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
print("Cosine similarity for Row ID 1 (Prompt vs Option A):", round(sim, 4))

Cosine similarity for Row ID 1 (Prompt vs Option A): 0.2328


## **Question 06**

Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [9]:
correct_matches = 0

for idx, row in train_df.iterrows():
    prompt_text = str(row["prompt"])
    options = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]
    tfidf_matrix = tfidf.transform([prompt_text] + options)
    sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:6])[0]
    best_idx = np.argmax(sims)
    best_option_letter = chr(ord("A") + best_idx)
    if best_option_letter == row["answer"]:
        correct_matches += 1

percentage = (correct_matches / len(train_df)) * 100
print(f"Percentage of instances where highest cosine similarity matches the correct answer: {percentage:.2f}%")

Percentage of instances where highest cosine similarity matches the correct answer: 13.70%


## **Question 07**

If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

In [10]:
def map_at_3(truth, preds):
    if truth == preds[0]:
        return 1.0
    elif truth == preds[1]:
        return 1/2
    elif truth == preds[2]:
        return 1/3
    else:
        return 0.0

score = map_at_3("C", ["C", "A", "B"])
print("MAP@3 score if truth is C and predictions are [C, A, B]:", score)

MAP@3 score if truth is C and predictions are [C, A, B]: 1.0


## **Question 08**

If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?

In [11]:
score = map_at_3("B", ["D", "B", "E"])
print("MAP@3 score if truth is B and predictions are [D, B, E]:", score)

MAP@3 score if truth is B and predictions are [D, B, E]: 0.5


## **Question 09**

The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [12]:
top_3_answers = train_df["answer"].value_counts().index[:3].tolist()
print("Top 3 most frequent answers:", top_3_answers)

map3_scores_majority = []
for truth in train_df["answer"]:
    map3_scores_majority.append(map_at_3(truth, top_3_answers))

print("Overall MAP@3 score of the Majority Class baseline:", np.mean(map3_scores_majority))

Top 3 most frequent answers: ['B', 'C', 'A']
Overall MAP@3 score of the Majority Class baseline: 0.42125


## **Question 10**

The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [13]:
map3_scores_tfidf = []
option_letters = ["A", "B", "C", "D", "E"]

for idx, row in train_df.iterrows():
    truth = row["answer"]
    prompt_text = str(row["prompt"])
    options = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]
    tfidf_matrix = tfidf.transform([prompt_text] + options)
    sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:6])[0]

    sorted_options = [opt for _, opt in sorted(zip(sims, option_letters), reverse=True)]
    top_3_preds = sorted_options[:3]
    map3_scores_tfidf.append(map_at_3(truth, top_3_preds))

print("Final average MAP@3 score of the TF-IDF pipeline:", np.mean(map3_scores_tfidf))

Final average MAP@3 score of the TF-IDF pipeline: 0.2763333333333333
